In [ ]:
### Cell 1 - Importing Libraries

import os
import pandas as pd
import numpy as np
import requests
import json
from typing import Dict, Any, Optional, List, Union
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import seaborn as sns
from sklearn.cluster import KMeans
import pybaseball as bb
import datetime
import time
import concurrent.futures

In [ ]:
# Cell 2 - Defining Functions to Get Pitching Data

# Option 1: Get MLB Stats API data only
# Option 2: Get Statcast data only
# Option 3: Get comprehensive data both datasets

def get_pitcher_stats(player_id: int, season: int) -> Dict[str, Any]:
    """
    Download the pitching statistics for a specific pitcher in a given season using MLB Stats API.
    
    Args:
        player_id (int): The MLB player ID of the pitcher
        season (int): The season year (e.g., 2023)
    
    Returns:
        Dict[str, Any]: Dictionary containing the pitcher's statistics
    """
    # MLB Stats API endpoint for player stats
    url = f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats"
    
    # Parameters for the API request - only get pitching stats
    params = {
        "stats": "season",
        "group": "pitching",  # Only request pitching stats
        "season": str(season),
        "sportId": "1",  # MLB is sportId 1
        "hydrate": "metrics"  # Include additional pitching metrics
    }
    
    try:
        # Make the API request
        response = requests.get(url, params=params)
        response.raise_for_status()  # Raise an exception for bad responses
        
        # Check if player is a pitcher
        data = response.json()
        if not data.get('stats') or not any(stat.get('group', {}).get('displayName') == 'pitching' 
                                          for stat in data.get('stats', [])):
            raise ValueError(f"Player {player_id} does not have pitching statistics")
            
        return data
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching pitching data: {e}")
        return {}
    except ValueError as e:
        print(e)
        return {}

def get_player_info(player_id: int) -> Dict[str, Any]:
    """
    Get basic information about a player.
    
    Args:
        player_id (int): The MLB player ID
    
    Returns:
        Dict[str, Any]: Dictionary containing the player's information
    """
    url = f"https://statsapi.mlb.com/api/v1/people/{player_id}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching player info: {e}")
        return {}

def process_stats(stats_data: Dict[str, Any], category: str) -> Optional[pd.DataFrame]:
    """
    Process the statistics data for a specific category (hitting, pitching, fielding).
    
    Args:
        stats_data (Dict[str, Any]): The statistics data from the API
        category (str): The category of statistics ('hitting', 'pitching', or 'fielding')
    
    Returns:
        Optional[pd.DataFrame]: DataFrame containing the processed statistics, or None if not available
    """
    if not stats_data or 'stats' not in stats_data:
        return None
    
    stats_list = []
    
    # Process all splits and groups to find the category data
    for stat_group in stats_data['stats']:
        group_name = stat_group.get('group', {}).get('displayName', '').lower()
        if group_name == category:
            if 'splits' in stat_group and stat_group['splits']:
                for split in stat_group['splits']:
                    if 'stat' in split:
                        stats_list.append(split['stat'])
    
    if stats_list:
        return pd.DataFrame(stats_list)
    
    return None

def get_all_player_stats(player_id: int, season: int) -> Dict[str, pd.DataFrame]:
    """
    Get all statistics (hitting, pitching, fielding) for a player in a specific season.
    
    Args:
        player_id (int): The MLB player ID
        season (int): The season year
    
    Returns:
        Dict[str, pd.DataFrame]: Dictionary with keys 'hitting', 'pitching', 'fielding' 
                               and corresponding DataFrames as values
    """
    stats_data = get_pitcher_stats(player_id, season)
    player_info = get_player_info(player_id)
    
    result = {}
    
    # Add player basic info
    if player_info and 'people' in player_info and player_info['people']:
        person = player_info['people'][0]
        result['player_info'] = pd.DataFrame([{
            'fullName': person.get('fullName', ''),
            'primaryNumber': person.get('primaryNumber', ''),
            'primaryPosition': person.get('primaryPosition', {}).get('abbreviation', ''),
            'birthDate': person.get('birthDate', ''),
            'currentTeam': person.get('currentTeam', {}).get('name', '') if 'currentTeam' in person else '',
            'height': person.get('height', ''),
            'weight': person.get('weight', '')
        }])
    
    # Process statistics for each category
    for category in ['pitching']:
        df = process_stats(stats_data, category)
        if df is not None:
            result[category] = df
    
    return result

def get_pitcher_statcast(pid: int, year: int) -> pd.DataFrame:
    """
    Get Statcast data for a specific pitcher in a given year.
    
    Args:
        pid (int): The MLB player ID of the pitcher
        year (int): The season year (e.g., 2024)
    
    Returns:
        pd.DataFrame: DataFrame containing the pitcher's Statcast data, or None if no data found/error occurs
    """
    try:
        # Add retry logic
        max_retries = 3
        for attempt in range(max_retries):
            try:
                # Convert year to date range
                start_date = f"{year}-03-01"
                end_date = f"{year}-11-01"
                
                print(f"Fetching Statcast data for pitcher ID {pid} in {year}...")
                
                # Get Statcast data for the pitcher
                df_pitches = bb.statcast_pitcher(start_dt=start_date,
                                       end_dt=end_date,
                                       player_id=pid)
                
                # Add pitcher ID as a column
                if df_pitches is not None and not df_pitches.empty:
                    df_pitches['pitcher_id'] = pid
                    print(f"Successfully retrieved Statcast data for pitcher ID {pid}")
                    return df_pitches
                else:
                    print(f"No Statcast data found for pitcher ID {pid}")
                    return None
                    
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"Attempt {attempt+1} failed: {e}. Retrying...")
                    time.sleep(2)  # Wait before retry
                else:
                    raise e
                    
    except Exception as e:
        print(f"Error getting Statcast data for pitcher ID {pid}: {e}")
        return None

def display_stats(stats_dict: Dict[str, pd.DataFrame]) -> None:
    """
    Display the statistics in a readable format.
    
    Args:
        stats_dict (Dict[str, pd.DataFrame]): Dictionary of statistics DataFrames
    """
    if 'player_info' in stats_dict:
        info = stats_dict['player_info'].iloc[0]
        print(f"Player: {info['fullName']} (#{info.get('primaryNumber', 'N/A')})")
        print(f"Position: {info.get('primaryPosition', 'N/A')}")
        print(f"Team: {info.get('currentTeam', 'N/A')}")
        print(f"Height: {info.get('height', 'N/A')} | Weight: {info.get('weight', 'N/A')} | DOB: {info.get('birthDate', 'N/A')}")
        print("\n" + "=" * 50 + "\n")
    
    categories = [cat for cat in stats_dict.keys() if cat != 'player_info']
    
    if not categories:
        print("No statistics available for this player in the specified season.")
        return
    
    for category in categories:
        print(f"{category.upper()} STATISTICS:")
        if stats_dict[category].empty:
            print(f"No {category} statistics available.")
        else:
            # Select the most important columns for pitching
            if category == 'pitching':
                columns = ['gamesPlayed', 'gamesStarted', 'inningsPitched', 'wins', 'losses', 'saves', 
                          'era', 'strikeOuts', 'whip', 'hitsPer9Inn', 'homeRunsPer9', 'strikeoutsPer9Inn']
            else:
                columns = stats_dict[category].columns
            
            # Display only columns that exist in the DataFrame
            existing_columns = [col for col in columns if col in stats_dict[category].columns]
            if existing_columns:
                print(stats_dict[category][existing_columns].to_string(index=False))
            else:
                print(stats_dict[category].to_string(index=False))
        
        print("\n" + "=" * 50 + "\n")

def save_stats_to_csv(stats_dict: Dict[str, pd.DataFrame], player_id: int, season: int) -> None:
    """
    Save the statistics to CSV files.
    
    Args:
        stats_dict (Dict[str, pd.DataFrame]): Dictionary of statistics DataFrames
        player_id (int): The MLB player ID
        season (int): The season year
    """
    player_name = "unknown"
    if 'player_info' in stats_dict and not stats_dict['player_info'].empty:
        player_name = stats_dict['player_info']['fullName'].iloc[0].replace(" ", "_")
    
    for category, df in stats_dict.items():
        if category != 'player_info':
            filename = f"{player_name}_{player_id}_{category}_{season}.csv"
            df.to_csv(filename, index=False)
            print(f"Saved {category} statistics to {filename}")

def save_statcast_to_csv(df_statcast: pd.DataFrame, player_id: int, season: int, player_name: str = "unknown") -> None:
    """
    Save Statcast data to CSV file.
    
    Args:
        df_statcast (pd.DataFrame): Statcast DataFrame
        player_id (int): The MLB player ID
        season (int): The season year
        player_name (str): Player's name for filename
    """
    if df_statcast is not None and not df_statcast.empty:
        filename = f"{player_name.replace(' ', '_')}_{player_id}_statcast_{season}.csv"
        df_statcast.to_csv(filename, index=False)
        print(f"Saved Statcast data to {filename}")

def get_comprehensive_pitcher_data(player_id: int, season: int) -> Dict[str, Any]:
    """
    Get both MLB Stats API data and Statcast data for a pitcher.
    
    Args:
        player_id (int): The MLB player ID
        season (int): The season year
    
    Returns:
        Dict[str, Any]: Dictionary containing both types of data
    """
    print(f"\nFetching comprehensive data for pitcher ID {player_id} in the {season} season...\n")
    
    # Get MLB Stats API data
    stats_dict = get_all_player_stats(player_id, season)
    
    # Get Statcast data
    df_statcast = get_pitcher_statcast(player_id, season)
    
    return {
        'stats_api_data': stats_dict,
        'statcast_data': df_statcast,
        'player_id': player_id,
        'season': season
    }

In [ ]:
# Retrieving the pitch data from the API

# Use pitcher_id from player's MLB.com page. Ex: https://www.mlb.com/player/max-fried-608331 --> Use 608331
pitcher_id = 554430
# Use the year of the season you want to get data for
year = 2022



comprehensive_data = get_comprehensive_pitcher_data(pitcher_id, year)
df_statcast = comprehensive_data['statcast_data']  # Statcast data stored here
df_statcast.head(10)

In [ ]:
# Create a new DataFrame with rows where "In_Zone_Our_Calc" is not equal to "In_Zone"
df_in_zone_discrepancies = df_statcast[df_statcast['In_Zone_Our_Calc'] != df_statcast['In_Zone']]

import matplotlib.pyplot as plt

# Plot the pitches with discrepancies between "In_Zone_Our_Calc" and "In_Zone"
fig, ax = plt.subplots(figsize=(6, 8))

# Plot all pitches as background (optional, for context)
ax.scatter(df_statcast['plate_x'], df_statcast['plate_z'], color='gray', alpha=0.5, label='All pitches')

# Plot pitches where "In_Zone_Our_Calc" != "In_Zone"
ax.scatter(
    df_in_zone_discrepancies['plate_x'],
    df_in_zone_discrepancies['plate_z'],
    c='red', label='Discrepancy', edgecolor='k', s=50
)

# Draw the strike zone (Statcast convention: plate_x in [-0.83, 0.83] feet)
# For vertical, use a typical strike zone (e.g., 1.5 to 3.5 feet), or show a box for a sample batter
strike_zone_left = -0.83
strike_zone_right = 0.83
strike_zone_bottom = 1.5
strike_zone_top = 3.5

# Draw rectangle for the strike zone
rect = plt.Rectangle(
    (strike_zone_left, strike_zone_bottom),
    strike_zone_right - strike_zone_left,
    strike_zone_top - strike_zone_bottom,
    fill=False, color='blue', linewidth=2, label='Strike Zone'
)
ax.add_patch(rect)

ax.set_xlabel('plate_x (feet)')
ax.set_ylabel('plate_z (feet)')
ax.set_title('Pitch Locations with In_Zone Discrepancies')
ax.set_xlim(-2, 2)
ax.set_ylim(0, 5)
ax.legend()
plt.show()


In [ ]:
# Create a new DataFrame with rows where "In_Zone_Our_Calc" is not equal to "In_Zone"
df_in_zone_strikes = df_statcast[
    (df_statcast['In_Zone'] == 1) & (df_statcast['In_Zone'] != df_statcast['In_Zone_Our_Calc'])]

df_in_zone_balls = df_statcast[
    (df_statcast['In_Zone'] == 0) & (df_statcast['In_Zone'] != df_statcast['In_Zone_Our_Calc'])]

import matplotlib.pyplot as plt

# Plot the pitches with discrepancies between "In_Zone_Our_Calc" and "In_Zone"
fig, ax = plt.subplots(figsize=(6, 8))

# Plot all pitches as background (optional, for context)
ax.scatter(df_statcast['plate_x'], df_statcast['plate_z'], color='gray', alpha=0.5, label='All pitches')

# Plot pitches where "In_Zone_Our_Calc" != "In_Zone", but In_Zone is 1
ax.scatter(
    df_in_zone_strikes['plate_x'],
    df_in_zone_strikes['plate_z'],
    c='blue', label='Discrepancy, In_Zone is Strike', edgecolor='k', s=50
)

# Plot pitches where "In_Zone_Our_Calc" != "In_Zone", but In_Zone is 0
ax.scatter(
    df_in_zone_balls['plate_x'],
    df_in_zone_balls['plate_z'],
    c='yellow', label='Discrepancy, In_Zone is Ball', edgecolor='k', s=50
)

# Draw the strike zone (Statcast convention: plate_x in [-0.83, 0.83] feet)
# For vertical, use a typical strike zone (e.g., 1.5 to 3.5 feet), or show a box for a sample batter
strike_zone_left = -0.83
strike_zone_right = 0.83
strike_zone_bottom = 1.5
strike_zone_top = 3.5

# Draw rectangle for the strike zone
rect = plt.Rectangle(
    (strike_zone_left, strike_zone_bottom),
    strike_zone_right - strike_zone_left,
    strike_zone_top - strike_zone_bottom,
    fill=False, color='blue', linewidth=2, label='Strike Zone'
)
ax.add_patch(rect)

ax.set_xlabel('plate_x (feet)')
ax.set_ylabel('plate_z (feet)')
ax.set_title('Pitch Locations with In_Zone Discrepancies')
ax.set_xlim(-2, 2)
ax.set_ylim(0, 5)
ax.legend()
plt.show()